In [36]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px
import re
from sklearn.feature_selection import mutual_info_classif
import joblib

In [2]:
df = pd.read_csv(r'D:\IT\Python\sklearn\hotel-booking-project\data\hotel_bookings.csv')
pd.set_option('display.max_columns', None)

In [3]:
df.head(3)

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02


In [34]:
df = df.drop(columns=["reservation_status","reservation_status_date"])

In [14]:
df['children'] = df['children'].fillna(0)
df['country'] = df['country'].fillna('Unknown')
df['has_company'] = df['company'].notna().astype(int)
df['has_agent'] = df['agent'].notna().astype(int)
df = df.drop('company', axis=1)
df = df.drop('agent', axis=1)

In [35]:
from sklearn.preprocessing import LabelEncoder

df_mi = df.copy()

for col in df_mi.select_dtypes(include="object"):
    df_mi[col] = LabelEncoder().fit_transform(
        df_mi[col].astype(str)
    )
X = df_mi.drop("is_canceled", axis=1)
y = df_mi["is_canceled"]

mi = mutual_info_classif(
    X,
    y,
    random_state=42
)

mi_scores = pd.Series(
    mi,
    index=X.columns
).sort_values(ascending=False)

print(mi_scores.head(20))

C:\Users\bben2\AppData\Local\Temp\ipykernel_20208\1931116648.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_mi.select_dtypes(include="object"):


deposit_type                   0.133761
lead_time                      0.084008
adr                            0.078469
country                        0.070934
market_segment                 0.043910
previous_cancellations         0.041365
total_of_special_requests      0.037910
room_changed                   0.037372
previous_bookings              0.029527
required_car_parking_spaces    0.028910
distribution_channel           0.023773
assigned_room_type             0.022728
booking_changes                0.018393
has_agent                      0.017245
customer_type                  0.016600
hotel                          0.014362
days_in_waiting_list           0.014241
total_nights                   0.013998
adults                         0.011939
total_guests                   0.011213
dtype: float64


In [24]:
df["total_guests"] = (
    df["adults"] +
    df["children"] +
    df["babies"]
)
df["total_nights"] = (
    df["stays_in_week_nights"] +
    df["stays_in_weekend_nights"]
)
df["room_changed"] = (
    df["reserved_room_type"]
    !=
    df["assigned_room_type"]
).astype(int)
df["previous_bookings"] = (
    df["previous_cancellations"]
    +
    df["previous_bookings_not_canceled"]
)

In [31]:
df.groupby("is_canceled")["adr"].describe()

,count,mean,std,min,25%,50%,75%,max
is_canceled,,,,,,,,
0,75166.0,99.987693,49.206263,-6.38,67.500,92.5,125.00,510.0
1,44224.0,104.964333,52.571142,0.00,72.415,96.2,127.62,5400.0
